In [1]:
import scanpy as sc
import pandas as pd
import loompy
import scvelo as scv



In [2]:
adata_sobj = sc.read('/Users/alexiscooper/The Francis Crick Dropbox/GuillemotF/Dropbox Alexis Cooper/Cooper_et_al_2025/4dpi/Preperation for R_to_python/h5ad_and_h5Seurat/4dpi_A2N2_5.h5ad')

In [3]:
adata_sobj

AnnData object with n_obs × n_vars = 6406 × 23726
    obs: 'orig_ident', 'nCount_RNA', 'nFeature_RNA', 'experiment', 'Path', 'RV', 'sample_type', 'timepoint', 'time', 'percent_mt', 'miQC_probability', 'miQC_keep', 'percent_rb', 'scDblFinder_class', 'S_Score', 'G2M_Score', 'Phase', 'Mscarlet3LTR', 'Mscarlet3LTR_1', 'RNA_snn_res_0_1', 'seurat_clusters', 'Outlier', 'mdist', 'GFPU3', 'Mscarlet', 'EGFPtetO', 'Mscarlet_1', 'EGFPtetO_1', 'NeonGreen', 'NeonGreen_1', 'GFPU3_1', 'new_names', 'sample_name', 'Sample', 'Barcode', 'cell_name_loom', 'RNA_snn_res_3', 'RNA_snn_res_0_05', 'cell_type', 'RNA_snn_res_2', 'Pseudotime_1', 'Pseudotime_2', 'Pseudotime_3', 'Pseudotime_4', 'Pseudotime_5', 'Pseudotime_6', 'Pseudotime_7', 'Pseudotime_9', 'Pseudotime_8', 'Pseudotime_10', 'Pseudotime_avg'
    var: 'vf_vst_counts.Ascl1SA6_mean', 'vf_vst_counts.Ascl1SA6_variance', 'vf_vst_counts.Ascl1SA6_variance.expected', 'vf_vst_counts.Ascl1SA6_variance.standardized', 'vf_vst_counts.Ascl1SA6_variable', 'vf_vst_coun

In [4]:
scaled_matrix = pd.read_csv(
    '/Users/alexiscooper/The Francis Crick Dropbox/GuillemotF/Dropbox Alexis Cooper/Cooper_et_al_2025/4dpi/Preperation for R_to_python/gene_counts_matrix/A2N2_5_data_210525.csv', ###or use scaled
    index_col=0)
scaled_matrix = scaled_matrix.T  # Transpose to match cells x genes format

In [5]:
adata_sobj = adata_sobj[:, adata_sobj.var_names.isin(scaled_matrix.columns)].copy()

adata_sobj.X = scaled_matrix.values

In [6]:
# Read loom-based data generated from velocyto output
adata_loom = sc.read_loom('/Users/alexiscooper/The Francis Crick Dropbox/GuillemotF/Dropbox Alexis Cooper/Cooper_et_al_2025/4dpi/Preperation for R_to_python/loom files/mergedLOOM_4DPI.loom')

In [7]:
# Make var_names unique to solve the write issue with h5ad
adata_loom.var_names_make_unique()

In [8]:
# Rename columns in the raw var DataFrame
if adata_sobj.raw is not None:
    adata_sobj.raw.var.rename(columns={'_index': 'features'}, inplace=True)



In [9]:
# Number of cell id entries from Seurat object not found in Loom
len(set(adata_sobj.obs_names) - set(adata_loom.obs_names))

1872

In [10]:
# Compare a few cell IDs between Seurat and Loom
print(adata_sobj.obs_names[:10])
print(adata_loom.obs_names[:10])


Index(['A4_cellranger_output:AAACCCAAGGCTAACGx-A4',
       'A4_cellranger_output:AAACCCACACAGCCTGx-A4',
       'A4_cellranger_output:AAACCCACATCGTGGCx-A4',
       'A4_cellranger_output:AAACCCATCAGACATCx-A4',
       'A4_cellranger_output:AAACGAAAGCTCGCACx-A4',
       'A4_cellranger_output:AAACGAATCCGAGAAGx-A4',
       'A4_cellranger_output:AAACGCTCAAAGACGCx-A4',
       'A4_cellranger_output:AAAGGATGTGCGTGCTx-A4',
       'A4_cellranger_output:AAAGGATTCGAGTACTx-A4',
       'A4_cellranger_output:AAAGGGCTCACCTGTCx-A4'],
      dtype='object')
Index(['A4_cellranger_output:AAAGGATTCGAGTACTx-A4',
       'A4_cellranger_output:AAAGGGCTCACCTGTCx-A4',
       'A4_cellranger_output:AACAAAGTCCATTTGTx-A4',
       'A4_cellranger_output:AACGTCAAGTAAGGGAx-A4',
       'A4_cellranger_output:AACAAAGGTAGATTGAx-A4',
       'A4_cellranger_output:AATAGAGAGGTTGTTCx-A4',
       'A4_cellranger_output:AAACGAATCGGCTTCTx-A4',
       'A4_cellranger_output:AAGTCGTTCCCGAGGTx-A4',
       'A4_cellranger_output:AAGTACCGTTAA

In [11]:
# Number of cell id entries from Loom not found in Seurat 
len(set(adata_loom.obs_names) - set(adata_sobj.obs_names))

59632

In [12]:
# Number of gene id entries from Seurat object not found in Loom
len(set(adata_sobj.var_names) - set(adata_loom.var_names))

19

In [13]:
# Number of gene id entries from Loom not found in Seurat
len(set(adata_loom.var_names) - set(adata_sobj.var_names))

8578

In [14]:
adata_loom.obs_names

Index(['A4_cellranger_output:AAAGGATTCGAGTACTx-A4',
       'A4_cellranger_output:AAAGGGCTCACCTGTCx-A4',
       'A4_cellranger_output:AACAAAGTCCATTTGTx-A4',
       'A4_cellranger_output:AACGTCAAGTAAGGGAx-A4',
       'A4_cellranger_output:AACAAAGGTAGATTGAx-A4',
       'A4_cellranger_output:AATAGAGAGGTTGTTCx-A4',
       'A4_cellranger_output:AAACGAATCGGCTTCTx-A4',
       'A4_cellranger_output:AAGTCGTTCCCGAGGTx-A4',
       'A4_cellranger_output:AAGTACCGTTAAGACAx-A4',
       'A4_cellranger_output:AAGTCGTTCGGTGCACx-A4',
       ...
       'A18_cellranger_output:TTTGGAGGTGCAATGGx-A18',
       'A18_cellranger_output:TTTACTGCATTCTCCGx-A18',
       'A18_cellranger_output:TTGGATGTCGCTTTATx-A18',
       'A18_cellranger_output:TTGGGTATCGGCTTCTx-A18',
       'A18_cellranger_output:TTTCACATCGCCGAGTx-A18',
       'A18_cellranger_output:TTGCGTCGTATCAAGAx-A18',
       'A18_cellranger_output:TTGATGGGTCGTAATCx-A18',
       'A18_cellranger_output:TTGGTTTGTTTACTGGx-A18',
       'A18_cellranger_output:TTGCATT

In [15]:
adata_sobj.obs_names

Index(['A4_cellranger_output:AAACCCAAGGCTAACGx-A4',
       'A4_cellranger_output:AAACCCACACAGCCTGx-A4',
       'A4_cellranger_output:AAACCCACATCGTGGCx-A4',
       'A4_cellranger_output:AAACCCATCAGACATCx-A4',
       'A4_cellranger_output:AAACGAAAGCTCGCACx-A4',
       'A4_cellranger_output:AAACGAATCCGAGAAGx-A4',
       'A4_cellranger_output:AAACGCTCAAAGACGCx-A4',
       'A4_cellranger_output:AAAGGATGTGCGTGCTx-A4',
       'A4_cellranger_output:AAAGGATTCGAGTACTx-A4',
       'A4_cellranger_output:AAAGGGCTCACCTGTCx-A4',
       ...
       'A9_cellranger_output:TTTGATCAGGGTTAATx-A9',
       'A9_cellranger_output:TTTGATCGTCATGCATx-A9',
       'A9_cellranger_output:TTTGATCGTGACATCTx-A9',
       'A9_cellranger_output:TTTGGAGCAAGCTGTTx-A9',
       'A9_cellranger_output:TTTGGAGCATCGTGGCx-A9',
       'A9_cellranger_output:TTTGGAGTCGCTAGCGx-A9',
       'A9_cellranger_output:TTTGGTTAGGTTATAGx-A9',
       'A9_cellranger_output:TTTGGTTCAGCCCACAx-A9',
       'A9_cellranger_output:TTTGGTTGTCAGGCAAx-A9',
 

In [16]:
# Filter to common cells and genes
common_cells = list(set(adata_sobj.obs_names) & set(adata_loom.obs_names))
common_genes = list(set(adata_sobj.var_names) & set(adata_loom.var_names))

adata_sobj_filtered = adata_sobj[common_cells, common_genes]
adata_loom_filtered = adata_loom[common_cells, common_genes]

# Merge both matrices
adata_merged = scv.utils.merge(adata_sobj_filtered, adata_loom_filtered)

# Adding layers from loom object including spliced and unspliced
adata_merged.layers['spliced'] = adata_loom_filtered.layers['spliced']
adata_merged.layers['unspliced'] = adata_loom_filtered.layers['unspliced']

# Check the merged object
adata_merged

AnnData object with n_obs × n_vars = 4534 × 23707
    obs: 'orig_ident', 'nCount_RNA', 'nFeature_RNA', 'experiment', 'Path', 'RV', 'sample_type', 'timepoint', 'time', 'percent_mt', 'miQC_probability', 'miQC_keep', 'percent_rb', 'scDblFinder_class', 'S_Score', 'G2M_Score', 'Phase', 'Mscarlet3LTR', 'Mscarlet3LTR_1', 'RNA_snn_res_0_1', 'seurat_clusters', 'Outlier', 'mdist', 'GFPU3', 'Mscarlet', 'EGFPtetO', 'Mscarlet_1', 'EGFPtetO_1', 'NeonGreen', 'NeonGreen_1', 'GFPU3_1', 'new_names', 'sample_name', 'Sample', 'Barcode', 'cell_name_loom', 'RNA_snn_res_3', 'RNA_snn_res_0_05', 'cell_type', 'RNA_snn_res_2', 'Pseudotime_1', 'Pseudotime_2', 'Pseudotime_3', 'Pseudotime_4', 'Pseudotime_5', 'Pseudotime_6', 'Pseudotime_7', 'Pseudotime_9', 'Pseudotime_8', 'Pseudotime_10', 'Pseudotime_avg', 'batch', 'initial_size_unspliced', 'initial_size_spliced', 'initial_size'
    var: 'vf_vst_counts.Ascl1SA6_mean', 'vf_vst_counts.Ascl1SA6_variance', 'vf_vst_counts.Ascl1SA6_variance.expected', 'vf_vst_counts.Ascl1

In [17]:
print("cells %d"%adata_merged.n_obs)

cells 4534


In [18]:
print(adata_merged)
print(adata_merged.obs.head())  # Check metadata
print(adata_merged.var.head())  # Check gene features
print(adata_merged.layers.keys())  # Confirm available layers


AnnData object with n_obs × n_vars = 4534 × 23707
    obs: 'orig_ident', 'nCount_RNA', 'nFeature_RNA', 'experiment', 'Path', 'RV', 'sample_type', 'timepoint', 'time', 'percent_mt', 'miQC_probability', 'miQC_keep', 'percent_rb', 'scDblFinder_class', 'S_Score', 'G2M_Score', 'Phase', 'Mscarlet3LTR', 'Mscarlet3LTR_1', 'RNA_snn_res_0_1', 'seurat_clusters', 'Outlier', 'mdist', 'GFPU3', 'Mscarlet', 'EGFPtetO', 'Mscarlet_1', 'EGFPtetO_1', 'NeonGreen', 'NeonGreen_1', 'GFPU3_1', 'new_names', 'sample_name', 'Sample', 'Barcode', 'cell_name_loom', 'RNA_snn_res_3', 'RNA_snn_res_0_05', 'cell_type', 'RNA_snn_res_2', 'Pseudotime_1', 'Pseudotime_2', 'Pseudotime_3', 'Pseudotime_4', 'Pseudotime_5', 'Pseudotime_6', 'Pseudotime_7', 'Pseudotime_9', 'Pseudotime_8', 'Pseudotime_10', 'Pseudotime_avg', 'batch', 'initial_size_unspliced', 'initial_size_spliced', 'initial_size'
    var: 'vf_vst_counts.Ascl1SA6_mean', 'vf_vst_counts.Ascl1SA6_variance', 'vf_vst_counts.Ascl1SA6_variance.expected', 'vf_vst_counts.Ascl1

In [19]:
adata_merged.var = adata_merged.var.rename(columns={'_index': 'index'})

In [20]:
adata_merged.write('/Users/alexiscooper/The Francis Crick Dropbox/GuillemotF/Dropbox Alexis Cooper/Cooper_et_al_2025/4dpi/Preperation for R_to_python/adata_and_loomLmerged/SandL_merged_A2N2_5_logNorm_2.h5ad', compression='gzip')